# Multi-UAV Swarm Trajectory Planning

**Pipeline overview**

| Cell | Stage | Modules |
|------|-------|---------|
| 1 | Environment setup & validation | `environment`, `visualizer_static` |
| 2 | Global path planning (A\*) | `global_planner` |
| 3 | Safe corridor extraction | `safe_corridor` |
| 4 | Individual trajectory generation (STO) | `trajectory_generator` |
| 5 | Conflict detection | `conflict_resolver` |
| 6 | Conflict resolution & replanning | `conflict_resolver` |
| 7 | Static 3-D visualisation | `visualizer_static` |
| 8 | Web simulation (HTML export) | `visualizer_web` |

In [1]:
# ── Cell 1 — Environment + Fleet setup ──────────────────────────────────────
import os, sys, logging

# ── sys.path setup ────────────────────────────────────────────────────────────
_SWARM_DIR  = os.path.abspath('')          # …/uavsafeplanning/SWARM
_PARENT_DIR = os.path.dirname(_SWARM_DIR)  # …/uavsafeplanning
for _p in (_SWARM_DIR, _PARENT_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── Imports ───────────────────────────────────────────────────────────────────
from environment       import Environment, ConfigValidationError
from uav               import Fleet
from visualizer_static import plot_environment_3d

# Silence sub-module DEBUG/INFO — only warnings/errors reach the output.
# Must come AFTER imports so the handlers are already registered.
for _mod in ('environment', 'uav'):
    logging.getLogger(_mod).setLevel(logging.WARNING)

# ── Config paths ──────────────────────────────────────────────────────────────
ENV_CONFIG   = os.path.join(_SWARM_DIR, 'configs', 'environment.yaml')
FLEET_CONFIG = os.path.join(_SWARM_DIR, 'configs', 'fleet.yaml')

def _tag(ok): return '✓' if ok else '✗'

# ── Environment ───────────────────────────────────────────────────────────────
try:
    env        = Environment.from_yaml(ENV_CONFIG)
    voxel_grid = env.to_voxel_grid()   # cached — A* (Cell 2) reuses this
    w = env.world
    print(f"✓  Environment  [{os.path.basename(ENV_CONFIG)}]  "
          f"{len(env.cylinders)} cylinders · {len(env.walls)} walls · "
          f"grid {w.nx}×{w.ny}×{w.nz}")
except ConfigValidationError as exc:
    print(f"✗  Environment  [{os.path.basename(ENV_CONFIG)}]  FAILED → {exc}")
    raise

# ── Fleet ─────────────────────────────────────────────────────────────────────
try:
    fleet = Fleet.from_yaml(FLEET_CONFIG)
    ids   = ', '.join(u.id for u in fleet.uavs)
    print(f"✓  Fleet        [{os.path.basename(FLEET_CONFIG)}]  "
          f"{len(fleet.uavs)} UAVs  ({ids})")
except ConfigValidationError as exc:
    print(f"✗  Fleet        [{os.path.basename(FLEET_CONFIG)}]  FAILED → {exc}")
    raise

# ── Position validation ───────────────────────────────────────────────────────
try:
    fleet.validate_with_environment(env)
    print("✓  Positions    all start / goal positions are obstacle-free")
except ConfigValidationError as exc:
    print(f"✗  Positions    FAILED → {exc}")
    raise

# ── Visualisation ─────────────────────────────────────────────────────────────
plot_environment_3d(env, fleet=fleet)


✓  Environment generated successfully.
────────────────────────────────────────────────────────────────
  ENVIRONMENT SUMMARY
────────────────────────────────────────────────────────────────
  World bounds:
    X : [0.0, 50.0] m
    Y : [0.0, 50.0] m
    Z : [0.0, 20.0] m
    Resolution : 0.5 m/voxel
    Grid size  : 100 × 100 × 40 = 400,000 voxels

  Cylinders (4):
    [cyl_0               ]  center=(10.0, 12.0)  r=2.00 m  h=15.00 m
    [cyl_1               ]  center=(38.0, 38.0)  r=2.50 m  h=18.00 m
    [cyl_2               ]  center=(15.0, 40.0)  r=1.50 m  h=12.00 m
    [cyl_3               ]  center=(42.0, 10.0)  r=2.00 m  h=14.00 m
  Walls (2):
    [passage_wall_south  ]  corners=[[22.0, 0.0], [26.0, 0.0], [26.0, 22.0], [22.0, 22.0]]  h=15.00 m
    [passage_wall_north  ]  corners=[[22.0, 28.0], [26.0, 28.0], [26.0, 50.0], [22.0, 50.0]]  h=15.00 m
────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────
  FLEE